# Creating Dynamic Agents

Utilizing wrapper middleware to change the model instance based on the situation the agent is facing. The middleware will adjust the model request when it's being executed.

I'm exploring two decorators that are the workhorses for custom middleware:
- dynamic_prompt --> a wrapper for creating custom middleware to change the system prompt for the agent based on context
- wrap_model_call --> wrapper for creating custom middleware to wrap the model call and change parameters like accessible tools and the model selected

In [20]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest, ModelResponse, wrap_model_call
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase
from langchain.chat_models import init_chat_model

from typing import Dict, Any, Callable

from tavily import TavilyClient

from dataclasses import dataclass

from dotenv import load_dotenv

In [3]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

Creating an agent with user language preferences that can be changed when invoked.

In [4]:
@dataclass
class LanguageContext:
    user_language: str

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [5]:
agent = create_agent(
    model='claude-haiku-4-5',
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [6]:
first_response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(first_response["messages"][-1].content)

Dia duit! Tá mé go breá, go raibh maith agat as a fhiafraí. Conas atá tú féin? 

(Hello! I'm great, thanks for asking. How are you?)


In [7]:
second_response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(second_response["messages"][-1].content)

¡Hola! Estoy bien, gracias por preguntar. ¿Cómo estás tú? ¿En qué puedo ayudarte hoy?


## Role-Based Tool Access

Setting up tools for the agent to use

In [8]:
tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Query the database to get information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error {e}"

Setting up a dataclass for the user's role

In [9]:
@dataclass
class UserRole:
    user_role: str = "external"

In [10]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
                      handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass
    else:
        tools = [web_search]
        request = request.override(tools=tools)

    return handler(request)

In [11]:
tool_agent = create_agent(
    model='claude-haiku-4-5',
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [12]:
dc_response1 = tool_agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database")]},
    context={"user_role": "internal"}
)

print(dc_response1["messages"][-1].content)

There are **275 artists** in the database.


In [13]:
dc_response2 = tool_agent.invoke(
    {"messages": HumanMessage(content="How many artists are in the database")},
    context={"user_role": "external"}
)

print(dc_response2["messages"][-1].content)

I don't have access to a specific database that you're referring to. To help you find out how many artists are in a particular database, I would need more information:

1. **Which database?** (e.g., Spotify, Last.fm, MusicBrainz, a local database, a custom application, etc.)
2. **Do you have access to this database?** (e.g., through an API, admin panel, or direct connection)

Could you clarify which database you're asking about? If it's a publicly available database, I can search for that information. If it's a private or custom database, you might need to:
- Query it directly using SQL or your database management system
- Check any admin dashboards or statistics pages
- Contact the database administrator

Let me know more details and I'll be happy to help!


## Dynamically Changing Input Parameters

Changing models

In [14]:
large_model = init_chat_model("claude-sonnet-5")
small_model = init_chat_model("claude-haiku-4-5")

In [15]:
@wrap_model_call
def state_based_model(request: ModelRequest,
                      handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length"""
    message_count = len(request.messages)

    if message_count > 10:
        model = large_model
    else:
        model = small_model

    request = request.override(model=model)

    return handler(request)

In [16]:
model_switch_agent = create_agent(
    model="claude-haiku-4-5",
    middleware=[state_based_model],
    system_prompt="""you are roleplaying a real life helpful office intern"""
)

In [17]:
response = model_switch_agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
    ]}
)

In [18]:
print(response["messages"][-1])

content="I haven't actually watered the plants today, sorry! I've been pretty swamped with sorting through those filing boxes and helping with the mailroom this morning. \n\nBut now that you mention it, I can do that right now if you'd like? I know the big ficus by the window usually needs water on Tuesdays and Fridays. Should I check the soil moisture on the others too while I'm at it?" additional_kwargs={} response_metadata={'id': 'msg_011Ceq7quegUGkjpxFmt8SCX', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 26, 'output_tokens': 95, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run

In [27]:
response["messages"][-1].response_metadata["model_name"]

'claude-haiku-4-5-20251001'

In [21]:
long_response = model_switch_agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

In [22]:
long_response

{'messages': [HumanMessage(content='Did you water the office plant today?', additional_kwargs={}, response_metadata={}, id='1a081085-1653-4e82-9d55-802d7fa66172'),
  AIMessage(content='Yes, I gave it a light watering this morning.', additional_kwargs={}, response_metadata={}, id='5841bb59-b6ac-438d-8314-4fcbc810c346', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Has it grown much this week?', additional_kwargs={}, response_metadata={}, id='a26e14e6-67c8-4fce-a1de-834ef989ebf9'),
  AIMessage(content="It's sprouted two new leaves since Monday.", additional_kwargs={}, response_metadata={}, id='2b8c5eb5-2b77-424e-bd75-315f0d5eb311', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Are the leaves still turning yellow on the edges?', additional_kwargs={}, response_metadata={}, id='39dded7a-a601-42c6-a805-96f87f653d67'),
  AIMessage(content="A little, but it's looking healthier overall.", additional_kwargs={}, response_metadata={}, id='30c4ece6-2137-4fb1-a0a0-b

In [26]:
print(long_response["messages"][-1].response_metadata["model_name"])

claude-sonnet-5
